### RNEMD Implementation
Implementation of Reverse Non-Equilibrium Molecular Dynamics to calculate Thermal Conductivity based on paper by Muller-Plathe
##### Algorithm (brief):
Müller‑Plathe RNEMD induces a steady-state heat flux by periodically exchanging velocities between the hottest particle in a “cold” slab and the coldest particle in a “hot” slab. Measure the steady temperature gradient and known imposed heat flux to compute thermal conductivity via Fourier’s law.

Citation: Florian Müller-Plathe; A simple nonequilibrium molecular dynamics method for calculating the thermal
        conductivity. J. Chem. Phys. 8 April 1997; 106 (14): 6082–6085. https://doi.org/10.1063/1.473271

### Overview
RNEMD is provided as a driver class to run reverse non‑equilibrium molecular dynamics on a single‑system, single‑element SimState. The class partitions the simulation box into slabs along the transport direction and enforces an exchange of kinetic energy between selected hot/cold slabs to induce a steady heat flux.

In [1]:
import numpy as np
import torch
import torch_sim as ts
from torch_sim.workflows.RNEMD import  RNEMD

Warp DeprecationWarning: The symbol `warp.vec` will soon be removed from the public API. Use `warp.types.vector` instead.


____
### Initializing the Simulation Setup
Two approaches:

1. Direct SimState input (use an already-built SimState)
2. Use RNEMD.create_simple_system(...) to generate a randomized single-element SimState

Notes on current implementation constraints:

- simstate.n_systems must be 1 (multi-system ensembles are not supported).
- Single element only: velocity swaps assume equal mass. Swapping velocities between different‑mass species does not generally conserve kinetic energy because KE = ½ m v^2; assigning the same velocity to unequal masses changes total kinetic energy and thus spuriously creates or destroys energy.
- nslabs must be even and > 0: the algorithm places the hot slab at index N/2 with slabs indexed 0..N−1; N must be even so that the opposite slab pairing is integer and symmetric.

#### Approach 1: Initialising RNEMD object using SimState (Direct SimState Input)


In [2]:
# Approach 1: Direct SimState Input
# existing_simstate: SimState = ...
# rnemd_setup = RNEMD(system_state=existing_simstate, nslabs=20)

### Approach 2: Create a simple system with RNEMD.create_simple_system()

RNEMD.create_simple_system() returns a SimState built from an ASE Atoms object with uniformly random positions.

In [3]:
# Approach 2: Using class method to create a simple system.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
argon_simstate = RNEMD.create_simple_system(
    atomic_number=18,                                        # Single element system
    box_dimensions_angs=(34.2498735, 68.499747, 102.749961), # (x,y,z)Dimensions of the simulation_box in Angstroms
    n_atoms=2592,                                            # number of atoms you want in the system
    pbc = [False, False, True],                              # optional, defaults to [False, False, True]
    device = device,                                         # optional, defaults to whichever device is available
    dtype=torch.float64                                      # optional, defaults to float32
)
rnemd_setup = RNEMD(system_state=argon_simstate, nslabs=20)  #nslabs defaults to 20

Initialization details (RNEMD.__init__)

Parameters:
- system_state (SimState): single‑system, single‑element SimState with a diagonal cell.
- nslabs (int): even number of slabs (>0) along transport direction.

Checks & effects:
- Errors: multiple systems or multiple species → NotImplementedError; nslabs ≤0 or odd → ValueError; non‑diagonal cell → ValueError.
- Sets internal attributes (n_atoms, atomic_number, atomic_mass_amu, device, dtype, nslabs, box dims) and calls _generate_system_slabs().

Rationale:
- Single system: multi‑system support not implemented.
- Single element: swapping velocities between different masses does not conserve kinetic energy.
- Even nslabs: hot slab is at index N/2 (slabs 0..N−1), so N must be even for a well‑defined opposite slab.

_____

___
### Setting up the interatomic model

Provide an object implementing ModelInterface (torch_sim.models.interface.ModelInterface). The model can be Lennard‑Jones or any model that implements forces/energies and integrates with ts.integrate.

Example Lennard‑Jones model setup (units converted for the model):

In [4]:
# define your model:
import ase
epsilon_J, sigma_m = 1.6512577588e-21, 3.405e-10 #in Joules and meters respectively
epsilon_eV, sigma_A = epsilon_J * 6.241509e18,  sigma_m * 1e10
cutoff_distance_m, cutoff_distance_A = 3 * sigma_m, 3 * sigma_A
atom_mass_amu = ase.data.atomic_masses[18]
atom_mass_kg = atom_mass_amu * 1.660539e-27  # amu_to_kg
simulation_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from torch_sim.models.lennard_jones import LennardJonesModel
lj_model = LennardJonesModel(
    sigma= sigma_A, # Å, 3.405
    epsilon=epsilon_eV,  # eV, 0.01030634
    cutoff=cutoff_distance_A, # Angstrom 10.215
    device=simulation_device,
    # technically this is true by default but mentioning this to make the code easier to read
    compute_forces=True,
    compute_stress=True,
    per_atom_energies=True,
    per_atom_stresses=True,
    use_neighbor_list= True,
)

___
### Running the simulation
High-level call (RNEMD.run_simulation). Example configuration and run:

In [5]:
# run the simulation:
from math import sqrt
timestep_reduced = 6.965e-3
timestep_s = timestep_reduced * sqrt(atom_mass_kg * sigma_m ** 2 / epsilon_J)
timestep_ps = timestep_s / 1e-12
simulation_timestep = timestep_ps
print(timestep_ps)
# argon_simstate = RNEMD.create_simple_system(18, (34.2498735, 2*34.2498735, 102.749961), )
instance = RNEMD(system_state=argon_simstate)

instance.run_simulation(
    model=lj_model,
    timestep_ps=simulation_timestep,
    W=15,
    nsteps_total=2_400,
    temp_K=8,
    n_equil_step=5000,
    perform_equilibration=True,
    perform_geometry_optimization=True,
)

0.015031518718469734
(02-23 13:41) torch_sim.workflows.RNEMD: Logging to /home/edraval/W26/test-world/RNEMD_simulation_data/simulation_logs.log
(02-23 13:41) torch_sim.workflows.RNEMD: Performing Geometry Optimization...
(02-23 13:45) torch_sim.workflows.RNEMD: [Geometry optimization finished] Post-Relaxation Energy is -175.86 eV
(02-23 13:45) torch_sim.workflows.RNEMD: Performing Equilibration...


/home/edraval/W26/test-world/.venv/lib/python3.12/site-packages/torch_sim/workflows/RNEMD.py:350: UserWarning: All systems have reached the maximum number of steps: 10000.
  relaxed_state = ts.runners.optimize(


(02-23 13:47) torch_sim.workflows.RNEMD: [Equilibration Completed] Post-Equilibration System Temperature: 7.93 K
(02-23 13:47) torch_sim.workflows.RNEMD: Simulation Data filepath set to /home/edraval/W26/test-world/RNEMD_simulation_data/simulation_data.h5
(02-23 13:47) torch_sim.workflows.RNEMD: Performing simulation...
(02-23 13:48) torch_sim.workflows.RNEMD: Simulation Concluded.


### Complete parameter summary:

1. **model (ModelInterface)**: interatomic model passed to ts.integrate.
2. **timestep_ps (float)**: integration timestep in picoseconds.
3. **W (int)**: apply system modifiers (e.g., velocity exchanges) every W integration steps.
4. **nsteps_total (int)**: total number of integration steps to perform.
5. **temp_K (float)**: target temperature (Kelvin).
6. **perform_equilibration (bool)**: run equilibration phase before production when True.
7. **n_equil_step (int)**: number of equilibration steps.
8. **n_exchanges_per_step (int)**: number of swap pairs performed every W steps.
9. **store_equil_data (bool)**: write equilibration data to disk.
10. **data_folder_path_abs (PathLike)**: Directory where simulation files are saved. (defaults to {cwd}/RNEMD_simulation_data)
11. **equilibration_data_filename / simulation_data_filename**: output filenames.
12. **log_to_file (bool)**: whether to save logger output to file.

_Note_: **If data_folder_path_abs is omitted**, it defaults to a folder named RNEMD_simulation_data in the current working directory (os.getcwd()). You can also give a relative path (e.g., ./simulation_data) to create/use a folder named simulation_data in the current directory; the same syntax works on Windows, macOS, and Linux.
_Although the code works with relative paths, specifying absolute path is recommended._
___



### What happens during integration and why we store intermediate data

Velocity exchanges (system modifiers) occur inside the integration loop (ts.integrate) so they modify the system state during the run. Many derived quantities (e.g., slabwise temperature gradient) are computed after integration and depend on stored observables from the run.

Implementation note:
- During integration, we store essential time-series: system temperature, potential energy, slabwise temperatures, and a separate intermediate file for velocity‑exchange bookkeeping (intmd_file). This separation avoids heavy modification to ts.integrate while still preserving enough information to compute heat fluxes and thermal conductivity during post‑processing.
- If save_simulation_results is True, velocity-exchange info is merged into the final {simulation_data_filename}.h5 during post-processing.

___


### Post-Simulation processing to calculate thermal

This will compute running averages, assemble exchanged-velocity histories, compute time-dependent temperature gradients, and optionally save full processed results to disk.

In [6]:
### Post-Processing the simulation to get the recorded values/properties
instance.post_simulation_processing(
    compute_running_values=True,
    save_simulation_results=True,
)

(02-23 13:48) torch_sim.workflows.RNEMD: Performing post-simulation processing...
(02-23 13:48) torch_sim.workflows.RNEMD: Unable to find slabwise_temperature property in acceptable_property_names
(02-23 13:48) torch_sim.workflows.RNEMD: Calculating running values...
(02-23 13:48) torch_sim.workflows.RNEMD: Storing calculated simulation quantities in /home/edraval/W26/test-world/RNEMD_simulation_data/simulation_data.h5...
(02-23 13:48) torch_sim.workflows.RNEMD: Deleting the intermediate file: /home/edraval/W26/test-world/RNEMD_simulation_data/intmd_vexchange_data.h5
(02-23 13:48) torch_sim.workflows.RNEMD: Storing running values in /home/edraval/W26/test-world/RNEMD_simulation_data/simulation_data.h5
(02-23 13:48) torch_sim.workflows.RNEMD: Post-Simulation Processing Complete!


___
### Retrieving stored/calculated values

Use instance.simulation_file_read_property(simulation_file, key) or RNEMD.simulation_file_read_property(simulation_file, key). (it's a static method)
n_exchanges_total = nsteps_total // W

Properties Recorded During Simulation:
- "system_temperature": System's temperature calculated using torch_sim.quantities.calc_temperature() (Shape: (n_exchange_total+1,))
- "system_potential": System's Potential Energy calculated using model(state)['energy'] (Shape: (n_exchanges_total+1, 1))
- "slabwise_temperature": Slabwise Temperature calculated using RNEMD._calculate_slabwise_temperature() (Shape: (n_exchanges_total+1, nslabs)) \
_Note_: The "+1" is because the ts.integrate reports (stored state and properties to record) once before integration.

If save_simulation_results=True:
- **v_hot_list**: hot‑velocity exchanges per exchange step; shape (n_exchanges_total, n_exchanges_per_step).
- **v_cold_list**: cold‑velocity exchanges; shape (n_exchanges_total, n_exchanges_per_step).
- **v_hot_cumulative**: cumulative hot‑exchange tensor used for pairwise statistics; shape (n_exchanges_per_step, n_exchanges_per_step, n_exchanges_total, n_exchanges_total). Zero‑padded as needed.
- **v_cold_cumulative**: cumulative cold‑exchange tensor; same shape as v_hot_cumulative.
- **dTdz_over_time**: instantaneous temperature gradient (per exchange step, per slab); shape (n_exchanges_total, nslabs).
- **dTdz_mean_z**: time‑averaged slabwise temperature gradient/profile (mean over time, per slab); shape (nslabs,).
- **avg_dTdz**: scalar time‑averaged gradient $\langle \partial T/\partial z \rangle$.
- **final_thermal_conductivity**: scalar $\kappa$ at final step.

If compute_running_values=True (running statistics):
- **running_vhot_squared**: pairwise running squared hot‑velocity correlations; shape (n_exchanges_total, n_exchanges_total).
- **running_vcold_squared**: same for cold velocities; shape (n_exchanges_total, n_exchanges_total).
- **running_sum_vhot_squared**: running cumulative sum of v_hot_squared; shape (n_exchanges_total,).
- **running_sum_vcold_squared**: running cumulative sum of v_cold_squared; shape (n_exchanges_total,).
- **running_dTdz_over_time**: slabwise temperature gradient time series; shape (n_exchanges_total, nslabs).
- **running_dTdz_mean_z**: running per‑step mean slab profile used for intermediate fits; shape (n_exchanges_total, n_exchanges_total - 1).
- **running_avg_dTdz**: running average gradient across time; shape (n_exchanges_total,).
- **running_thermal_conductivity**: running κ time series; shape (n_exchanges_total,).

In [11]:
# Approach 1: Using self.simulation_file
# simulation_file = instance.simulation_file
# property = RNEMD.simulation_file_read_property(simulation_file, 'prop')

# Approach 2: Get the simulation file using simulation_parameters dictionary
simulation_file = instance.simulation_parameters['simulation_filepath']
final_kappa = instance.simulation_file_read_property(simulation_file, 'final_thermal_conductivity') # v_hot_cumulative
print("Final kappa = ", final_kappa.item(), "(in SI units)")
# Instance 3: 0.02 Instance 4: 0.002, Instance 1: 0.185 Instance 2: 0.2
# Approach 3: abs_path of the simulation file
# abs_filepath = '/home/...'
# RNEMD.simulation_file_read_property(abs_filepath, 'final_thermal_conductivity')
# store kinetic, potential, temperature distribution + std dev,

Final kappa =  0.09972009807825089 (in SI units)
